In [11]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2005_2006"
    / "Demographics"
)

xpt_file = DATA_DIR / "DEMO_D.xpt"
csv_file = DATA_DIR / "DEMO_D_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Keep untouched original for fidelity comparison
source_df = df.copy()


print("=" * 78)
print("NHANES 2005-2006 DEMO_D XPT -> CSV")
print("=" * 78)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "SDDSRVYR",
    "RIDSTATR",
    "RIDEXMON",
    "RIAGENDR",
    "RIDAGEYR",
    "RIDAGEMN",
    "RIDAGEEX",
    "RIDRETH1",
    "DMQMILIT",
    "DMDBORN",
    "DMDCITZN",
    "DMDYRSUS",
    "DMDEDUC3",
    "DMDEDUC2",
    "DMDSCHOL",
    "DMDMARTL",
    "DMDHHSIZ",
    "DMDFMSIZ",
    "INDHHINC",
    "INDFMINC",
    "INDFMPIR",
    "RIDEXPRG",
    "DMDHRGND",
    "DMDHRAGE",
    "DMDHRBRN",
    "DMDHREDU",
    "DMDHRMAR",
    "DMDHSEDU",
    "SIALANG",
    "SIAPROXY",
    "SIAINTRP",
    "FIALANG",
    "FIAPROXY",
    "FIAINTRP",
    "MIALANG",
    "MIAPROXY",
    "MIAINTRP",
    "AIALANG",
    "WTINT2YR",
    "WTMEC2YR",
    "SDMVPSU",
    "SDMVSTRA"
]


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "DEMO_D columns/order differ from expected.\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 10348:
    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 43:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: DEMO_D structure = 10,348 rows x 43 columns."
)


# ============================================================
# 4. KNOWN PANDAS/XPORT ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)

true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{int(true_zero_counts.sum()):,}"
)

print(
    "Tiny artifact occurrences:",
    f"{int(tiny_counts.sum()):,}"
)


print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 5. CDC-VALIDATED ZERO VARIABLES
# ============================================================

ZERO_ALLOWED_COLUMNS = [
    "RIDAGEYR",
    "RIDAGEMN",
    "RIDAGEEX",
    "DMDEDUC3",
    "INDFMPIR",
    "WTMEC2YR"
]


EXPECTED_TINY_COUNTS = {
    "RIDAGEYR": 526,
    "RIDAGEMN": 40,
    "RIDAGEEX": 16,
    "DMDEDUC3": 244,
    "INDFMPIR": 76,
    "WTMEC2YR": 398
}


# ============================================================
# 6. SAFETY CHECK
# ============================================================

actual_tiny_columns = set(
    tiny_counts[
        tiny_counts > 0
    ].index
)

allowed_tiny_columns = set(
    ZERO_ALLOWED_COLUMNS
)


unexpected = (
    actual_tiny_columns
    -
    allowed_tiny_columns
)


if unexpected:

    raise ValueError(
        "STOP: Tiny values found in unexpected "
        "DEMO_D variables:\n"
        + str(sorted(unexpected))
    )


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_count = int(
        tiny_counts.get(
            col,
            0
        )
    )

    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count} tiny values, "
            f"found {actual_count}."
        )


if int(tiny_counts.sum()) != 1300:

    raise ValueError(
        "Unexpected total number of tiny artifacts."
    )


print(
    "\nPASS: All 1,300 tiny values occur only "
    "in validated zero-permitted variables."
)


# ============================================================
# 7. EXTRA VALIDATION - WTMEC2YR
# ============================================================

#
# The 398 zero MEC weights must correspond exactly
# to the 398 participants who were interviewed only.
#

mec_zero_mask = (
    df["WTMEC2YR"]
    .eq(TINY_VALUE)
)

interview_only_mask = (
    df["RIDSTATR"]
    .eq(1)
)


if not np.array_equal(
    mec_zero_mask.to_numpy(),
    interview_only_mask.to_numpy()
):

    raise ValueError(
        "STOP: WTMEC2YR tiny values do not correspond "
        "exactly to RIDSTATR=1."
    )


print(
    "PASS: 398 WTMEC2YR zero artifacts correspond "
    "exactly to 398 interview-only participants."
)


# ============================================================
# 8. RESTORE TRUE ZEROS
# ============================================================

print("\n--- ZERO RESTORATION ---")


for col in ZERO_ALLOWED_COLUMNS:

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )

    count = int(
        mask.sum()
    )

    print(
        f"{col:<10}: {count:,} tiny values -> 0"
    )

    df.loc[
        mask,
        col
    ] = 0.0


# ============================================================
# 9. VERIFY ZERO COUNTS
# ============================================================

print("\n--- RESTORED ZERO COUNTS ---")


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_zero = int(
        df[col]
        .eq(0)
        .sum()
    )

    print(
        f"{col:<10}: {actual_zero:,}"
    )

    if actual_zero != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count} zeros, "
            f"found {actual_zero}."
        )


print(
    "PASS: All 1,300 zeros restored correctly."
)


# ============================================================
# 10. DEFINE REAL DECIMAL VARIABLES
# ============================================================

#
# IMPORTANT:
#
# These variables must NOT be converted to Int64.
#
# INDFMPIR = poverty-income ratio
# WTINT2YR = interview sample weight
# WTMEC2YR = MEC sample weight
#

DECIMAL_COLUMNS = [
    "INDFMPIR",
    "WTINT2YR",
    "WTMEC2YR"
]


# ============================================================
# 11. CHECK ALL OTHER FIELDS ARE WHOLE-NUMBER
# ============================================================

print("\n--- FRACTIONAL VALUE CHECK ---")


CODE_COLUMNS = [
    col
    for col in numeric_cols
    if col not in (
        ["SEQN"]
        + DECIMAL_COLUMNS
    )
]


fraction_failures = []


for col in ["SEQN"] + CODE_COLUMNS:

    values = (
        df[col]
        .dropna()
    )

    if values.empty:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional.any():

        fraction_failures.append(
            col
        )

        print(
            f"\nUnexpected decimals in {col}:"
        )

        print(
            values[
                fractional
            ].head(20)
        )


if fraction_failures:

    raise ValueError(
        "STOP: Unexpected decimal values found in "
        "whole-number fields:\n"
        + str(fraction_failures)
    )


print(
    "PASS: All identifier/code/count fields "
    "contain whole-number values."
)


# ============================================================
# 12. CONFIRM DECIMAL FIELDS
# ============================================================

print("\n--- DECIMAL FIELD CHECK ---")


for col in DECIMAL_COLUMNS:

    s = df[col]

    fractional_count = int(
        (
            s.dropna()
            !=
            np.round(
                s.dropna()
            )
        )
        .sum()
    )

    print(
        col,
        "| valid:",
        int(s.notna().sum()),
        "| missing:",
        int(s.isna().sum()),
        "| min:",
        s.min(),
        "| max:",
        s.max(),
        "| fractional:",
        fractional_count
    )


print(
    "PASS: Real decimal variables will remain floats "
    "and will not be rounded."
)


# ============================================================
# 13. CLEAN SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(df["SEQN"].isna().sum()) != 0
    or
    int(df["SEQN"].duplicated().sum()) != 0
    or
    df["SEQN"].min() != 31127
    or
    df["SEQN"].max() != 41474
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 14. FORMAT ONLY WHOLE-NUMBER FIELDS
# ============================================================

#
# NO .round()
#
# 4.0    -> 4
# 85.0   -> 85
# 9999.0 -> 9999
#
# But:
#
# INDFMPIR stays float
# WTINT2YR stays float
# WTMEC2YR stays float
#

for col in CODE_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: Whole-number DEMO_D fields formatted "
    "without unnecessary .0."
)


# ============================================================
# 15. CDC CHECK - DATA RELEASE NUMBER
# ============================================================

if (
    df["SDDSRVYR"]
    .value_counts()
    .to_dict()
    !=
    {4: 10348}
):

    raise ValueError(
        "SDDSRVYR differs from CDC."
    )


print(
    "PASS: SDDSRVYR = 4 for all 10,348 participants."
)


# ============================================================
# 16. CDC CHECK - INTERVIEW / EXAM STATUS
# ============================================================

EXPECTED_RIDSTATR = {
    1: 398,
    2: 9950
}


if (
    df["RIDSTATR"]
    .value_counts()
    .sort_index()
    .to_dict()
    !=
    EXPECTED_RIDSTATR
):

    raise ValueError(
        "RIDSTATR differs from CDC."
    )


print(
    "PASS: RIDSTATR matches CDC."
)


# ============================================================
# 17. CDC CHECK - EXAM PERIOD
# ============================================================

EXPECTED_RIDEXMON = {
    1: 4793,
    2: 5157
}


if not (
    df["RIDEXMON"]
    .value_counts()
    .sort_index()
    .to_dict()
    ==
    EXPECTED_RIDEXMON

    and

    int(
        df["RIDEXMON"]
        .isna()
        .sum()
    )
    ==
    398
):

    raise ValueError(
        "RIDEXMON differs from CDC."
    )


print(
    "PASS: RIDEXMON matches CDC."
)


# ============================================================
# 18. CDC CHECK - GENDER
# ============================================================

EXPECTED_RIAGENDR = {
    1: 5080,
    2: 5268
}


if (
    df["RIAGENDR"]
    .value_counts()
    .sort_index()
    .to_dict()
    !=
    EXPECTED_RIAGENDR
):

    raise ValueError(
        "RIAGENDR differs from CDC."
    )


print(
    "PASS: RIAGENDR matches CDC."
)


# ============================================================
# 19. CDC CHECK - AGE IN YEARS
# ============================================================

s = df["RIDAGEYR"]


if not (
    int(
        s.between(
            0,
            84
        ).sum()
    )
    ==
    10178

    and

    int(
        s.eq(85)
        .sum()
    )
    ==
    170

    and

    int(
        s.eq(0)
        .sum()
    )
    ==
    526

    and

    int(
        s.isna()
        .sum()
    )
    ==
    0
):

    raise ValueError(
        "RIDAGEYR differs from CDC/source."
    )


print(
    "PASS: RIDAGEYR matches CDC."
)


# ============================================================
# 20. CDC CHECK - AGE IN MONTHS
# ============================================================

s = df["RIDAGEMN"]


if not (
    int(
        s.between(
            0,
            1019
        ).sum()
    )
    ==
    10178

    and

    int(
        s.eq(0)
        .sum()
    )
    ==
    40

    and

    int(
        s.isna()
        .sum()
    )
    ==
    170
):

    raise ValueError(
        "RIDAGEMN differs from CDC/source."
    )


print(
    "PASS: RIDAGEMN matches CDC."
)


# ============================================================
# 21. CDC CHECK - EXAM AGE IN MONTHS
# ============================================================

s = df["RIDAGEEX"]


if not (
    int(
        s.between(
            0,
            1019
        ).sum()
    )
    ==
    9777

    and

    int(
        s.eq(0)
        .sum()
    )
    ==
    16

    and

    int(
        s.isna()
        .sum()
    )
    ==
    571
):

    raise ValueError(
        "RIDAGEEX differs from CDC/source."
    )


print(
    "PASS: RIDAGEEX matches CDC."
)


# ============================================================
# 22. CDC CHECK - RACE / ETHNICITY
# ============================================================

EXPECTED_RIDRETH1 = {
    1: 2847,
    2: 349,
    3: 3928,
    4: 2710,
    5: 514
}


if (
    df["RIDRETH1"]
    .value_counts()
    .sort_index()
    .to_dict()
    !=
    EXPECTED_RIDRETH1
):

    raise ValueError(
        "RIDRETH1 differs from CDC."
    )


print(
    "PASS: RIDRETH1 matches CDC."
)


# ============================================================
# 23. CDC CHECK - DMDEDUC3
# ============================================================

EXPECTED_DMDEDUC3 = {
    0: 244,
    1: 191,
    2: 190,
    3: 206,
    4: 206,
    5: 205,
    6: 290,
    7: 265,
    8: 300,
    9: 298,
    10: 307,
    11: 277,
    12: 48,
    13: 214,
    14: 10,
    15: 143,
    66: 37,
    99: 1
}


actual_dmdeduc3 = (
    df["DMDEDUC3"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if not (
    actual_dmdeduc3
    ==
    EXPECTED_DMDEDUC3

    and

    int(
        df["DMDEDUC3"]
        .isna()
        .sum()
    )
    ==
    6916
):

    print(
        "Actual DMDEDUC3:",
        actual_dmdeduc3
    )

    raise ValueError(
        "DMDEDUC3 differs from CDC."
    )


print(
    "PASS: DMDEDUC3 matches CDC."
)


# ============================================================
# 24. CDC CHECK - INDFMPIR
# ============================================================

s = df["INDFMPIR"]


if not (
    int(
        s.between(
            0,
            4.99
        ).sum()
    )
    ==
    8409

    and

    int(
        s.eq(5)
        .sum()
    )
    ==
    1405

    and

    int(
        s.eq(0)
        .sum()
    )
    ==
    76

    and

    int(
        s.isna()
        .sum()
    )
    ==
    534
):

    raise ValueError(
        "INDFMPIR differs from CDC/source."
    )


print(
    "PASS: INDFMPIR matches CDC/source."
)


# ============================================================
# 25. CDC CHECK - SAMPLE WEIGHTS
# ============================================================

if not (
    int(
        df["WTINT2YR"]
        .notna()
        .sum()
    )
    ==
    10348

    and

    int(
        df["WTMEC2YR"]
        .notna()
        .sum()
    )
    ==
    10348

    and

    int(
        df["WTMEC2YR"]
        .eq(0)
        .sum()
    )
    ==
    398
):

    raise ValueError(
        "Sample weight validation failed."
    )


print(
    "PASS: WTINT2YR / WTMEC2YR counts match CDC/source."
)


# ============================================================
# 26. BUILD EXPECTED SOURCE
# ============================================================

#
# Expected source =
#
# original XPT
# +
# ONLY the 6 validated zero restorations.
#

expected_df = source_df.copy()


for col in ZERO_ALLOWED_COLUMNS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 27. EXACT VALUE FIDELITY BEFORE EXPORT
# ============================================================

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


value_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    if (
        col == "SEQN"
        or
        col in CODE_COLUMNS
    ):

        current_values = (
            df[col]
            .astype("Float64")
            .to_numpy(
                dtype=float,
                na_value=np.nan
            )
        )

    else:

        current_values = (
            df[col]
            .to_numpy()
        )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        value_failures.append(
            col
        )


if value_failures:

    raise ValueError(
        "Unexpected value changes in:\n"
        + str(value_failures)
    )


print(
    "PASS: All 43 DEMO_D variables preserved exactly "
    "after validated zero restoration."
)


# ============================================================
# 28. FINAL TINY VALUE CHECK
# ============================================================

remaining_tiny = int(
    df.select_dtypes(
        include=[np.number]
    )
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print("\nRemaining tiny values:", remaining_tiny)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny-value artifacts remain."
    )


print(
    "PASS: No tiny-value artifact remains."
)


# ============================================================
# 29. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO float_format
# NO round()
# NO recalculation
#
# DECIMAL_COLUMNS remain floats.
# Whole-number code fields are Int64.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 30. RE-READ CSV FOR NUMERIC FIDELITY
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 31. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print("\n--- XPT -> CSV NUMERIC FIDELITY ---")


csv_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )

    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 43 CSV columns numerically match "
    "the expected XPT."
)


# ============================================================
# 32. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )

    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\nMissing-position mismatches:", missing_mismatches)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 33. RAW CSV TOKEN CHECK
# ============================================================

#
# IMPORTANT:
#
# Do NOT globally reject ".0".
#
# Real decimal fields may legitimately contain:
#
# INDFMPIR = 5.0 / 0.0
# WTMEC2YR = 0.0
#
# Only SEQN and code fields should not end in .0.
#

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)
    raw_rows = list(reader)


bad_code_dot_zero = 0


for row in raw_rows:

    for col in ["SEQN"] + CODE_COLUMNS:

        value = row[col]

        if value.endswith(".0"):

            bad_code_dot_zero += 1


print(
    "SEQN/code values ending '.0':",
    bad_code_dot_zero
)


if bad_code_dot_zero != 0:

    raise ValueError(
        "Unnecessary .0 formatting remains "
        "in identifier/code fields."
    )


print(
    "PASS: SEQN/code fields have no unnecessary .0."
)


# ============================================================
# 34. RAW DECIMAL FIDELITY
# ============================================================

#
# Compare decimal field CSV text directly against
# str(float(XPT value)) after validated zero restoration.
#

decimal_mismatches = {
    col: 0
    for col in DECIMAL_COLUMNS
}


for i, row in enumerate(raw_rows):

    for col in DECIMAL_COLUMNS:

        xpt_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(xpt_value)
            else str(float(xpt_value))
        )


        if row[col] != expected_text:

            decimal_mismatches[col] += 1


print(
    "Decimal raw-text mismatches:",
    decimal_mismatches
)


if any(
    decimal_mismatches.values()
):

    raise ValueError(
        "Decimal formatting changed during CSV export."
    )


print(
    "PASS: INDFMPIR, WTINT2YR, WTMEC2YR "
    "appear AS IS in CSV."
)


# ============================================================
# 35. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 36. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 78)
print("DEMO_D CONVERSION COMPLETE")
print("=" * 78)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))
print("Validated zero restorations: 1,300")
print("Real decimal variables:", DECIMAL_COLUMNS)
print("SEQN/code .0 formatting: 0 expected")
print("XPT -> CSV differences: 0 expected")
print("Missing-position differences: 0 expected")

NHANES 2005-2006 DEMO_D XPT -> CSV
Rows: 10,348
Columns: 43
PASS: DEMO_D structure = 10,348 rows x 43 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 1,300

Tiny values by variable:
RIDAGEYR    526
WTMEC2YR    398
DMDEDUC3    244
INDFMPIR     76
RIDAGEMN     40
RIDAGEEX     16

PASS: All 1,300 tiny values occur only in validated zero-permitted variables.
PASS: 398 WTMEC2YR zero artifacts correspond exactly to 398 interview-only participants.

--- ZERO RESTORATION ---
RIDAGEYR  : 526 tiny values -> 0
RIDAGEMN  : 40 tiny values -> 0
RIDAGEEX  : 16 tiny values -> 0
DMDEDUC3  : 244 tiny values -> 0
INDFMPIR  : 76 tiny values -> 0
WTMEC2YR  : 398 tiny values -> 0

--- RESTORED ZERO COUNTS ---
RIDAGEYR  : 526
RIDAGEMN  : 40
RIDAGEEX  : 16
DMDEDUC3  : 244
INDFMPIR  : 76
WTMEC2YR  : 398
PASS: All 1,300 zeros restored correctly.

--- FRACTIONAL VALUE CHECK ---
PASS: All identifier/code/count fields contain whole-number values.

--- DECIMAL FIELD CHECK 